Import the necessary libraries

In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import numpy as np
import re as re
from matplotlib import pyplot as plt
pd.options.display.max_rows = 999
pd.options.display.max_columns = 90

Load the dataset into sqlite3

In [ ]:
con = sqlite3.connect("bmarket.db")


In [ ]:
cursor = con.cursor()

        # Query the sqlite_master table to get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        
        # Fetch all results
table_names = [row[0] for row in cursor.fetchall()]

In [ ]:
print(table_names)

In [ ]:
query = "SELECT * FROM bank_marketing"  # Replace 'your_table_name' with the actual table name
df = pd.read_sql_query(query, con)
df.head(10)

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    print(f"\n🔹 {col} — {df[col].nunique()} unique values:")
    print(df[col].unique())

In [ ]:
df = df.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)

In [ ]:
def to_snake(name):
    return re.sub(r'[\s\.-]+', '_', name.strip().lower())

df = df.rename(columns=lambda c: to_snake(c))
df

In [ ]:
df.isna().sum()

In [ ]:
# Remove the word 'years' from the 'age' column and convert to numeric
if 'age' in df.columns:
    df['age'] = (
        df['age']
        .astype(str)
        .str.replace('years', '', case=False)
        .str.strip()
    )
    df['age'] = pd.to_numeric(df['age'], errors='coerce')

# Verify change
df['age'].head()

In [ ]:
df["occupation"] = df["occupation"].astype(str).str.replace(".", "", regex=False)
df['occupation'] = df['occupation'].astype(str).str.replace("-", "_", regex=False)
df["occupation"].value_counts(dropna=False)

In [ ]:
df["education_level"] = df["education_level"].astype(str).str.replace(".", "_", regex=False)
df["education_level"].value_counts(dropna=False)

In [ ]:
# Standarize the 
if 'contact_method' in df.columns:
    df['contact_method'] = df['contact_method'].replace(['cell'], 'cellular')
    df['contact_method'] = df['contact_method'].replace(['Telephone'], 'telephone')

# Check if replacement worked
df['contact_method'].value_counts()

In [ ]:
df.loc[df['campaign_calls'] < 0, 'campaign_calls'] = 0
df['campaign_calls'].value_counts(dropna=False)

Change all the unknown value to NULL

In [ ]:
df = df.replace("unknown", np.nan)
df = df.replace({None: np.nan})
##df["previous_contact_days"] = df["previous_contact_days"].replace(999, np.nan)
##df["age"] = df["age"].replace(150, np.nan)

In [ ]:
for col in df.columns:
    print(f"\n🔹 {col} — {df[col].nunique()} unique values:")
    print(df[col].unique())